# 05_shap_explainability.ipynb

## Explainable AI for Heart Disease Prediction

This notebook adds **SHAP (SHapley Additive exPlanations)** explainability to the trained heart disease prediction model.

### What is SHAP?
SHAP is a game-theoretic approach to explain the output of machine learning models. It tells us how much each feature contributes to pushing the model's prediction away from the base value (the average model output).

### Why Explainable AI Matters
- **Trust**: Doctors and patients need to understand why a model makes a prediction
- **Debugging**: Identify when the model makes unexpected decisions
- **Compliance**: Medical models often require interpretability (HIPAA, FDA regulations)
- **Improvement**: Understand which features matter most

### How SHAP Differs from Feature Importance
- **Feature Importance**: Global measure of which features the model uses most (e.g., how much each feature splits nodes in a tree)
- **SHAP Values**: Individual contribution of each feature to **each prediction** (local explanation)
- SHAP is based on game theory and guarantees fairness in contribution attribution

In [ ]:
# Import required libraries
import os
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Install SHAP if not already installed
# Uncomment the line below if SHAP is not installed
# !pip install shap

import shap

print(f'SHAP version: {shap.__version__}')
print('All libraries imported successfully!')

## Load trained model and data

In [ ]:
# Define paths to saved model, scaler, and cleaned data
best_model_path = os.path.join('models', 'best_model.pkl')
scaler_path = os.path.join('models', 'scaler.pkl')
clean_data_path = os.path.join('data', 'processed', 'cleaned_data.csv')
results_dir = os.path.join('results')

# Create results directory if it doesn't exist
os.makedirs(results_dir, exist_ok=True)

# Load the trained best model
if not os.path.exists(best_model_path):
    raise FileNotFoundError(f'Best model not found at {best_model_path}. Run model training first.')
best_model = joblib.load(best_model_path)
print(f'Loaded best model: {type(best_model).__name__}')

# Load the scaler
if not os.path.exists(scaler_path):
    raise FileNotFoundError(f'Scaler not found at {scaler_path}. Run feature selection first.')
scaler = joblib.load(scaler_path)
print('Loaded scaler')

# Load cleaned data
if not os.path.exists(clean_data_path):
    raise FileNotFoundError(f'Cleaned data not found at {clean_data_path}. Run preprocessing first.')
df = pd.read_csv(clean_data_path)
print(f'Loaded cleaned data with shape: {df.shape}')

## Prepare features and scale data

In [ ]:
# Define the same feature columns used during training
feature_columns = [
    'age_years', 'gender', 'height', 'weight', 'BMI',
    'ap_hi', 'ap_lo', 'cholesterol', 'gluc', 'smoke', 'alco', 'active'
]

# Create feature matrix
X = df[feature_columns].copy()

# Verify all required columns exist
missing_cols = [col for col in feature_columns if col not in df.columns]
if missing_cols:
    raise KeyError(f'Missing columns: {missing_cols}')

print(f'Feature columns ({len(feature_columns)}): {feature_columns}')
print(f'X shape: {X.shape}')

# Scale the features using the pre-fitted scaler
X_scaled = scaler.transform(X)
print(f'Scaled data shape: {X_scaled.shape}')

# Convert to DataFrame for easier interpretation
X_scaled_df = pd.DataFrame(X_scaled, columns=feature_columns, index=X.index)
print('\nFirst row of scaled data:')
print(X_scaled_df.iloc[0])

## Create SHAP Explainer

SHAP offers different explainers for different model types:
- **TreeExplainer**: For tree-based models (Random Forest, XGBoost, etc.)
- **KernelExplainer**: Model-agnostic (works with any model but slower)
- **DeepExplainer**: For deep learning models

We automatically detect the model type and create the appropriate explainer.

In [ ]:
# Auto-detect model type and create appropriate SHAP explainer
model_type = type(best_model).__name__
print(f'Model type: {model_type}')

# For tree-based models (Random Forest, XGBoost), use TreeExplainer
if model_type in ['RandomForestClassifier', 'XGBClassifier']:
    print(f'Creating SHAP TreeExplainer for {model_type}...')
    explainer = shap.TreeExplainer(best_model)
    print('TreeExplainer created successfully')
else:
    # Fallback to KernelExplainer for other model types
    print(f'Creating SHAP KernelExplainer for {model_type}...')
    # Use a small sample for background distribution to speed up computation
    background = shap.sample(X_scaled_df, 100, random_state=42)
    explainer = shap.KernelExplainer(best_model.predict_proba, background)
    print('KernelExplainer created successfully')

print(f'\nExplainer base value (average model output): {explainer.expected_value}')

## Compute SHAP values

SHAP values tell us how much each feature pushes the model's prediction away from the base value.
- Positive SHAP value: Feature increases disease risk
- Negative SHAP value: Feature decreases disease risk

In [ ]:
# Calculate SHAP values for all samples
# This may take a few minutes depending on dataset size
print('Computing SHAP values for all samples (this may take a few minutes)...')
shap_values = explainer.shap_values(X_scaled_df)

# For binary classification models, shap_values might be a list [class_0_values, class_1_values]
# We'll use class 1 (disease/positive class) values
if isinstance(shap_values, list):
    shap_values_disease = shap_values[1]  # disease class
    print(f'Binary classification detected. Using SHAP values for disease class (class 1)')
else:
    shap_values_disease = shap_values

print(f'SHAP values shape: {shap_values_disease.shape}')
print('SHAP values computed successfully!')

## Generate SHAP Visualizations

In [ ]:
# 1. SHAP Summary Plot (Beeswarm Plot)
# Shows how each feature value affects model output
print('Generating SHAP Summary Plot...')
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values_disease, X_scaled_df, show=False)
plt.title('SHAP Summary Plot: Feature Contributions to Heart Disease Prediction')
plt.tight_layout()
summary_plot_path = os.path.join(results_dir, 'shap_summary.png')
plt.savefig(summary_plot_path, dpi=300, bbox_inches='tight')
print(f'Saved SHAP Summary Plot to {summary_plot_path}')
plt.show()
plt.close()

In [ ]:
# 2. SHAP Bar Plot
# Shows mean absolute SHAP values (global feature importance)
print('\nGenerating SHAP Bar Plot...')
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values_disease, X_scaled_df, plot_type='bar', show=False)
plt.title('SHAP Bar Plot: Mean |SHAP| Values (Feature Importance)')
plt.tight_layout()
bar_plot_path = os.path.join(results_dir, 'shap_bar.png')
plt.savefig(bar_plot_path, dpi=300, bbox_inches='tight')
print(f'Saved SHAP Bar Plot to {bar_plot_path}')
plt.show()
plt.close()

## Analyze a single patient example

In [ ]:
# Select a sample patient for detailed explanation
# Let's pick the first patient with high disease risk (class 1 prediction)
sample_idx = 0

# Get model's prediction for this patient
X_sample = X_scaled_df.iloc[[sample_idx]]
prediction_proba = best_model.predict_proba(X_sample)[0]
predicted_class = best_model.predict(X_sample)[0]
disease_probability = prediction_proba[1]  # probability of class 1 (disease)

print(f'\n{"="*60}')
print(f'PATIENT SAMPLE #{sample_idx}')
print(f'{"="*60}')
print(f'\nPrediction: Class {predicted_class} (Disease: {"Yes" if predicted_class == 1 else "No"})')
print(f'Confidence: {disease_probability*100:.1f}% risk of heart disease')
print(f'\nUnscaled Patient Features:')
patient_features = X.iloc[sample_idx]
print(patient_features)

# Get SHAP values for this patient
patient_shap_values = shap_values_disease[sample_idx]
print(f'\nSHAP values for this patient:')
shap_df = pd.DataFrame({
    'Feature': feature_columns,
    'SHAP Value': patient_shap_values,
    'Original Value': X.iloc[sample_idx].values
}).sort_values('SHAP Value', ascending=False, key=abs)
print(shap_df)

In [ ]:
# 3. SHAP Waterfall Plot for the sample patient
# Shows how each feature contributes to the final prediction
print('\nGenerating SHAP Waterfall Plot for sample patient...')
plt.figure(figsize=(12, 8))

# Create explainer output for waterfall plot
explainer_output = shap.Explanation(
    values=shap_values_disease[sample_idx],
    base_values=explainer.expected_value if isinstance(explainer.expected_value, (int, float)) else explainer.expected_value[1],
    data=X_scaled_df.iloc[sample_idx].values,
    feature_names=feature_columns
)

shap.waterfall_plot(explainer_output, show=False)
plt.title(f'SHAP Waterfall Plot: Patient #{sample_idx} - Risk Breakdown')
plt.tight_layout()
waterfall_plot_path = os.path.join(results_dir, f'shap_waterfall_patient_{sample_idx}.png')
plt.savefig(waterfall_plot_path, dpi=300, bbox_inches='tight')
print(f'Saved SHAP Waterfall Plot to {waterfall_plot_path}')
plt.show()
plt.close()

In [ ]:
# 4. SHAP Force Plot for the sample patient (if supported)
# Shows a visual explanation of how features push the prediction
try:
    print('\nGenerating SHAP Force Plot for sample patient...')
    force_plot = shap.force_plot(
        explainer.expected_value if isinstance(explainer.expected_value, (int, float)) else explainer.expected_value[1],
        shap_values_disease[sample_idx],
        X_scaled_df.iloc[sample_idx],
        show=False
    )
    # Save force plot (Note: force plots are interactive in notebooks)
    print('Force plot generated successfully (interactive in notebook)')
except Exception as e:
    print(f'Force plot generation skipped: {str(e)}')

## Generate Human-Readable Explanation

In [ ]:
# Generate readable explanation for the sample patient
# Separate features into risk-increasing (positive SHAP) and protective (negative SHAP)

# Get top positive contributors (increase disease risk)
top_positive_idx = np.argsort(patient_shap_values)[-3:][::-1]  # top 3
top_positive_features = [(feature_columns[i], patient_shap_values[i], X.iloc[sample_idx, i]) for i in top_positive_idx if patient_shap_values[i] > 0]

# Get top negative contributors (protective factors)
top_negative_idx = np.argsort(patient_shap_values)[:3]  # bottom 3
top_negative_features = [(feature_columns[i], patient_shap_values[i], X.iloc[sample_idx, i]) for i in top_negative_idx if patient_shap_values[i] < 0]

# Determine risk level
if disease_probability >= 0.70:
    risk_level = 'HIGH RISK'
elif disease_probability >= 0.30:
    risk_level = 'MEDIUM RISK'
else:
    risk_level = 'LOW RISK'

# Print formatted explanation
explanation = f"""
{'='*70}
HEART DISEASE RISK PREDICTION - PATIENT #{sample_idx}
{'='*70}

PREDICTION: {risk_level} ({disease_probability*100:.1f}% probability of heart disease)

{'='*70}
TOP RISK FACTORS (Increase Disease Risk):
{'='*70}
"""

if top_positive_features:
    for feature, shap_val, original_val in top_positive_features:
        explanation += f"  • {feature:20s} = {original_val:8.2f}  (contribution: +{shap_val:.4f})\n"
else:
    explanation += "  (None - all features are protective)\n"

explanation += f"""
{'='*70}
PROTECTIVE FACTORS (Decrease Disease Risk):
{'='*70}
"""

if top_negative_features:
    for feature, shap_val, original_val in top_negative_features:
        explanation += f"  • {feature:20s} = {original_val:8.2f}  (contribution: {shap_val:.4f})\n"
else:
    explanation += "  (None - all features increase risk)\n"

explanation += f"""
{'='*70}
MODEL INTERPRETATION:
{'='*70}
This prediction is based on 12 medical features analyzed by the trained
heart disease prediction model. The SHAP values above show how much each
feature contributes to pushing the prediction toward disease risk.

IMPORTANT: This is a machine learning model prediction, not a medical
diagnosis. Always consult with a healthcare professional for medical advice.
{'='*70}
"""

print(explanation)

# Save explanation to file
explanation_path = os.path.join(results_dir, f'patient_{sample_idx}_explanation.txt')
with open(explanation_path, 'w') as f:
    f.write(explanation)
print(f'\nSaved explanation to {explanation_path}')

## Save SHAP Feature Importance to CSV

In [ ]:
# Calculate global SHAP feature importance (mean absolute SHAP values)
mean_abs_shap = np.abs(shap_values_disease).mean(axis=0)

# Create feature importance dataframe
feature_importance_df = pd.DataFrame({
    'Feature': feature_columns,
    'Mean |SHAP|': mean_abs_shap,
    'Rank': range(1, len(feature_columns) + 1)
}).sort_values('Mean |SHAP|', ascending=False).reset_index(drop=True)

# Update rank
feature_importance_df['Rank'] = range(1, len(feature_importance_df) + 1)

print('\nSHAP Feature Importance (Global):') 
print(feature_importance_df)

# Save to CSV
importance_csv_path = os.path.join(results_dir, 'shap_feature_importance.csv')
feature_importance_df.to_csv(importance_csv_path, index=False)
print(f'\nSaved feature importance to {importance_csv_path}')

## Summary: Key Insights from SHAP Analysis

In [ ]:
# Print overall summary
print(f"""
{'='*70}
SHAP EXPLAINABILITY ANALYSIS SUMMARY
{'='*70}

Model Type: {model_type}
Total Patients Analyzed: {len(X_scaled_df)}
Total Features: {len(feature_columns)}

Explainer Type: {'TreeExplainer' if model_type in ['RandomForestClassifier', 'XGBClassifier'] else 'KernelExplainer'}
Base Value (Average Model Output): {explainer.expected_value if isinstance(explainer.expected_value, (int, float)) else explainer.expected_value[1]:.4f}

TOP 5 MOST IMPORTANT FEATURES (SHAP):
{'='*70}
""")

for idx, row in feature_importance_df.head(5).iterrows():
    print(f"  {row['Rank']}. {row['Feature']:20s} Mean |SHAP|: {row['Mean |SHAP|']:.4f}")

print(f"""
{'='*70}
INTERPRETATION:
{'='*70}
- Features with higher Mean |SHAP| values have more influence on predictions
- Positive SHAP values → increase disease risk
- Negative SHAP values → decrease disease risk (protective)
- SHAP values are local (per-patient) explanations
- Feature importance is global (average across all patients)

OUTPUTS SAVED:
  ✓ {summary_plot_path}
  ✓ {bar_plot_path}
  ✓ {waterfall_plot_path}
  ✓ {importance_csv_path}
  ✓ {explanation_path}

{'='*70}
Explainability analysis complete!
{'='*70}
""")